# Electronic structure of MgO: band structure, DOS, and fat bands

*Based on Problems 9–11 of the [ICTP-MARVEL College 2026](https://github.com/marvel-nccr/ictp-marvel-college-2026) day-01 exercise, originally authored by Edward Linscott.*

This notebook uses the converged parameters from [`qe_convergence_tests.ipynb`](qe_convergence_tests.ipynb) and the equilibrium lattice parameter $a_0$ from [`qe_eos_bulkmodulus.ipynb`](qe_eos_bulkmodulus.ipynb) to compute the electronic structure of MgO.

1. **Problem 9** — band structure along the FCC Brillouin-zone path L → Γ → X → W → K → Γ.
2. **Problem 10** — total density of states (DOS) and orbital-projected DOS (PDOS).
3. **Problem 11** *(optional)* — fat bands: bands coloured by orbital character.

> [!NOTE]
> MgO is an ionic insulator. DFT-PBE underestimates the band gap of insulators (~5–6 eV computed vs ~7.8 eV experiment).

In [ ]:
from pathlib import Path
import glob, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
    ProjwfcNamelist,
)
from convergence_runner import QERunner, RY_TO_EV
from bandstructure_tools import (
    parse_fermi_energy, parse_hs_positions, parse_gamma_symmetries,
    read_bands_gnu, build_kpath_str, print_bands_summary,
    plot_band_structure, read_projwfc_weights,
)

RUN_ROOT   = Path('.').resolve()
PSEUDO_DIR = RUN_ROOT / 'pseudo'
BS_DIR     = RUN_ROOT / 'out' / 'bandstructure'
BS_DIR.mkdir(parents=True, exist_ok=True)

_pw_candidates = sorted(glob.glob('/home/pietro/repositories/q-e/build_*/bin/pw.x'))
if _pw_candidates:
    _bin_dir    = Path(_pw_candidates[0]).parent
    PW_CMD      = [str(_bin_dir / 'pw.x')]
    BANDS_CMD   = [str(_bin_dir / 'bands.x')]
    DOS_CMD     = [str(_bin_dir / 'dos.x')]
    PROJWFC_CMD = [str(_bin_dir / 'projwfc.x')]
else:
    PW_CMD      = [shutil.which('pw.x')]
    BANDS_CMD   = [shutil.which('bands.x')]
    DOS_CMD     = [shutil.which('dos.x')]
    PROJWFC_CMD = [shutil.which('projwfc.x')]
if not PW_CMD[0]:
    raise RuntimeError('pw.x not found.')

PSEUDOS = {
    'Mg': 'Mg.upf',
    'O':  'O.upf',
}

# Converged parameters from qe_convergence_tests.ipynb and qe_eos_bulkmodulus.ipynb
ECUTWFC = 60      # Ry
NK_SCF  = 8       # nk×nk×nk for SCF (FCC primitive cell)
A0      = 4.212   # Å — equilibrium lattice parameter from BM EOS fit; update with your value
NBND    = 16      # bands to compute (≥2× the 4 occupied bands)

## Problem 9: Band structure

The **band structure** $E_n(\mathbf{k})$ shows how electron energies vary across the Brillouin zone for each band index $n$. It reveals whether the gap is direct or indirect, how large it is, and which orbitals make up the band edges.

### Workflow

1. **SCF** — self-consistently converge the charge density on a regular **k**-mesh.
2. **bands** — fix the converged density and diagonalise the Hamiltonian on a dense **k**-path (`calculation = 'bands'`). No self-consistency is performed at the new **k**-points.
3. **bands.x** — post-process: sort bands by continuity and write a gnuplot-readable file (`*.dat.gnu`).

> [!NOTE]
> The `bands` run reuses the charge density stored in `prefix.save/` from the SCF. That directory must exist and must not have been overwritten (e.g. by a subsequent NSCF) before `bands.x` is called.

### FCC Brillouin zone path

MgO crystallises in the rocksalt structure with an FCC primitive cell (ibrav=2). The conventional high-symmetry path is:

$$\mathrm{L} \to \Gamma \to \mathrm{X} \to \mathrm{W} \to \mathrm{K} \to \Gamma$$

| Label | Crystal coordinates |
|-------|---------------------|
| L | $(½,\, ½,\, ½)$ |
| Γ | $(0,\; 0,\; 0)$ |
| X | $(½,\; 0,\; ½)$ |
| W | $(½,\; ¼,\; ¾)$ |
| K | $(⅜,\; ⅜,\; ¾)$ |

We set `nbnd = 16`. The pseudopotentials used here are ONCV norm-conserving: `Mg.upf` has $z_\text{val}=10$ (2s²2p⁶3s²) and `O.upf` has $z_\text{val}=6$ (2s²2p⁴), giving **8 occupied bands** (16 electrons / 2 per band). The gap separates band 8 (O 2p valence top) from band 9 (Mg 3s conduction bottom).

In [ ]:
FORCE_RERUN = False

atoms = bulk('MgO', 'rocksalt', a=A0)

scf_input = PWInput(
    ControlNamelist(calculation='scf', prefix='mgo_bs',
                    pseudo_dir=str(PSEUDO_DIR), outdir=str(BS_DIR)),
    SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ECUTWFC),
    ElectronsNamelist(),
    AtomicSpeciesCard.from_atoms(atoms, PSEUDOS),
    AtomicPositionsCard.from_atoms(atoms, units='crystal'),
    KPointsAutoCard(2, nk=NK_SCF),
)

scf_in  = BS_DIR / 'mgo_bs.scf.in'
scf_out = BS_DIR / 'mgo_bs.scf.out'

if FORCE_RERUN or not scf_out.exists():
    scf_in.write_text(scf_input.to_string())
    with open(scf_out, 'w') as fh:
        subprocess.run(PW_CMD + ['-input', str(scf_in)], stdout=fh, stderr=fh, check=True)

scf_stdout = scf_out.read_text()
E_F = parse_fermi_energy(scf_stdout)
print(f'SCF done.  Highest occupied level: E_F = {E_F:.4f} eV')

In [ ]:
FCC_PATH = [
    ('L', 0.5,   0.5,   0.5  ),
    ('G', 0.0,   0.0,   0.0  ),
    ('X', 0.5,   0.0,   0.5  ),
    ('W', 0.5,   0.25,  0.75 ),
    ('K', 0.375, 0.375, 0.75 ),
    ('G', 0.0,   0.0,   0.0  ),
]
NPT = [50, 60, 30, 20, 64]   # one per segment; total = 225 k-points

from types import SimpleNamespace
_kpath_card = SimpleNamespace(to_string=lambda: build_kpath_str(FCC_PATH, NPT))

The arc length of each segment is the Euclidean distance $|\Delta\mathbf{k}|$ between consecutive high-symmetry points, computed in Cartesian reciprocal space and expressed in units of $2\pi/a$. A crystal-coordinate vector $(k_1, k_2, k_3)$ maps to Cartesian via $\mathbf{k} = k_1\mathbf{b}_1 + k_2\mathbf{b}_2 + k_3\mathbf{b}_3$, where $\mathbf{b}_1, \mathbf{b}_2, \mathbf{b}_3$ are the primitive reciprocal lattice vectors of the FCC cell. For this path the lengths are: L–Γ = 0.866, Γ–X = 1.000, X–W = 0.500, W–K = 0.354, K–Γ = 1.061. `NPT` samples ~60 k-points per unit arc length so all segments look equally smooth in the plot — a uniform count would leave the long K–Γ segment under-sampled.

In [ ]:
bands_input = PWInput(
    # calculation='bands': fixes the SCF charge density and diagonalises the Hamiltonian
    # on the k-path only — no self-consistency is performed at the new k-points
    ControlNamelist(calculation='bands', prefix='mgo_bs',
                    pseudo_dir=str(PSEUDO_DIR), outdir=str(BS_DIR)),
    SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ECUTWFC, nbnd=NBND),
    ElectronsNamelist(),
    AtomicSpeciesCard.from_atoms(atoms, PSEUDOS),
    AtomicPositionsCard.from_atoms(atoms, units='crystal'),
    _kpath_card,
)

bands_in  = BS_DIR / 'mgo_bs.bands.in'
bands_out = BS_DIR / 'mgo_bs.bands.out'

if FORCE_RERUN or not bands_out.exists():
    bands_in.write_text(bands_input.to_string())
    with open(bands_out, 'w') as fh:
        subprocess.run(PW_CMD + ['-input', str(bands_in)], stdout=fh, stderr=fh, check=True)

print('bands pw.x done.')

In [ ]:
# bands.x reads the pw.x bands output, analyses symmetry, and writes *.dat.gnu for plotting
FILBAND      = str(BS_DIR / 'mgo_bs.bands.dat')
bands_pp_in  = BS_DIR / 'mgo_bs.bands_pp.in'
bands_pp_out = BS_DIR / 'mgo_bs.bands_pp.out'

bands_pp_in.write_text(
    f"&BANDS\n  prefix = 'mgo_bs'\n  outdir = '{BS_DIR}'\n"
    f"  filband = '{FILBAND}'\n  lsym = .true.\n/\n"
)

if FORCE_RERUN or not Path(FILBAND + '.gnu').exists():
    with open(bands_pp_in) as fin, open(bands_pp_out, 'w') as fout:
        subprocess.run(BANDS_CMD, stdin=fin, stdout=fout, stderr=fout, check=True)

print(f'bands.x done  →  {Path(FILBAND).name}.gnu')

In [ ]:
k_coords, bands_ev = read_bands_gnu(FILBAND + '.gnu')
hs_x       = parse_hs_positions(bands_pp_out.read_text())
sym_gamma  = parse_gamma_symmetries(bands_pp_out.read_text())

n_occ  = 8   # Mg.upf z_val=10 + O.upf z_val=6 → 16 e⁻/cell → 8 occupied bands
gap_ev = bands_ev[n_occ].min() - bands_ev[n_occ - 1].max()

print_bands_summary(k_coords, bands_ev, E_F, hs_x, FCC_PATH, n_occ)

print('\nBand symmetries at Γ (O_h point group):')
for s in sym_gamma:
    n1, n2 = s['bands']
    bstr = str(n1) if n1 == n2 else f'{n1}–{n2}'
    print(f'  band {bstr:4s}  {s["energy_ev"]:8.3f} eV   {s["label"]:8s}  (deg {s["deg"]})')

### Band symmetry at Γ

MgO has the rocksalt structure with full cubic symmetry (point group O$_h$, *m*-3*m*). At the Γ point the crystal eigenstates transform as irreducible representations (irreps) of O$_h$. The irrep label tells you the orbital character directly:

| Irrep | Character | Degeneracy |
|-------|-----------|------------|
| Γ$_1^+$ (A$_{1g}$) | *s*-like (totally symmetric) | 1 |
| Γ$_4^-$ (T$_{1u}$) | *p*-like | 3 |

The norm-conserving pseudopotentials used here treat the following states explicitly as valence:

| Pseudo | Config | $z_\text{val}$ | Semicore |
|--------|--------|----------------|---------|
| `Mg.upf` | 2s² 2p⁶ 3s² | 10 | 2s, 2p |
| `O.upf`  | 2s² 2p⁴     |  6 | 2s     |

Including 2s and 2p of Mg and 2s of O as valence gives **8 occupied bands** (16 electrons / 2 per band). The semicore states appear as flat, deep bands in the band structure. The O 2p valence manifold transforms as the 3-fold Γ$_4^-$; the Mg 3s conduction edge as Γ$_1^+$. The PDOS in Problem 10 will confirm this orbital character quantitatively.

In [ ]:
hs_labels = [r'$\Gamma$' if p[0] == 'G' else p[0] for p in FCC_PATH]
plot_band_structure(k_coords, bands_ev, E_F, hs_x, hs_labels, gap_ev)

<details>
<summary>Show plotting code</summary>

```python
fig, (ax_full, ax_gap) = plt.subplots(1, 2, figsize=(11, 5))

for ax in (ax_full, ax_gap):
    for band in bands_ev:
        ax.plot(k_coords, band - E_F, color='steelblue', lw=0.9)
    ax.axhline(0, color='gray', lw=0.7, ls='--')
    for x in hs_x[1:-1]:
        ax.axvline(x, color='k', lw=0.6)
    ax.set_xticks(hs_x)
    ax.set_xticklabels(hs_labels)
    ax.set_xlim(k_coords[0], k_coords[-1])
    ax.set_ylabel('$E - E_F$ (eV)')

ax_full.set_ylim(-80, 15)
ax_full.set_title('MgO bands — full range')

ax_gap.set_ylim(-25, 15)
ax_gap.set_title('MgO bands — valence & gap')
ax_gap.annotate(f'gap = {gap_ev:.2f} eV',
                xy=(hs_x[0] * 0.05 + hs_x[1] * 0.95, gap_ev / 2),
                fontsize=9, color='firebrick')

fig.tight_layout()
plt.show()
```

</details>

**Exercise.** Look at the full-range panel. The pseudopotential table above tells you which atomic states are treated explicitly.

1. How many distinct groups of bands can you count?
2. For each group: how many bands does it contain? Is it dispersive or flat? What is its approximate energy at Γ relative to $E_F$?
3. Associate each group to one of the atomic states listed in the pseudopotential table.

<details>
<summary>Solution</summary>

With 10 valence electrons on Mg (2s² 2p⁶ 3s²) and 6 on O (2s² 2p⁴), there are 16 electrons per primitive cell → 8 occupied bands, split into four groups:

| Bands | Energy at Γ | Count | State | Dispersive? |
|-------|-------------|-------|-------|-------------|
| 1     | ≈ −73 eV    | 1     | Mg 2s | no — core-like, nearly flat |
| 2–4   | ≈ −43 eV    | 3     | Mg 2p | no — flat, 3-fold degenerate at Γ |
| 5     | ≈ −19 eV    | 1     | O 2s  | no — flat |
| 6–8   | −5 to 0 eV  | 3     | O 2p  | yes — forms the valence band |

Bands 1–5 are **semicore**: deep atomic states barely perturbed by the crystal field. The O 2p manifold (bands 6–8) is the true valence band and disperses by ~5 eV across the BZ. The first conduction band (band 9 in the right panel) has Mg 3s character.

</details>

## Problem 10: Density of states

The **density of states** $N(E) = \sum_{n,\mathbf{k}} \delta(E - E_{n\mathbf{k}})$ counts electronic states per unit energy. The **projected DOS** (PDOS) decomposes $N(E)$ by orbital character via Löwdin projections onto atomic wavefunctions.

### Workflow

1. **Back up** `prefix.save/` — the NSCF will overwrite it (needed for Problem 11).
2. **NSCF** — reuse the SCF density; compute eigenvalues on a dense uniform **k**-mesh with `occupations = 'tetrahedra_opt'` (optimised tetrahedra integration, no smearing needed for an insulator).
3. **dos.x** — integrate $N(E)$ using tetrahedra weights.
4. **projwfc.x** — project onto atomic wavefunctions → one PDOS file per orbital.

> [!WARNING]
> `calculation = 'nscf'` overwrites `prefix.save/`. Run the backup cell before NSCF, or you will lose the bands wavefunction data needed for Problem 11.

### Expected orbital character in MgO

| Energy region | Character |
|---------------|-----------|
| Valence band top (−5 to 0 eV) | O $2p$ |
| Valence band bottom (~ −20 eV) | O $2s$ |
| Conduction band bottom (gap edge) | Mg $3s$ |

In [ ]:
NK_NSCF = 12

# Back up bands data before NSCF overwrites prefix.save/
save_dir     = BS_DIR / 'mgo_bs.save'
bands_backup = BS_DIR / 'mgo_bs_bands.save'
if save_dir.exists() and not bands_backup.exists():
    shutil.copytree(save_dir, bands_backup)
    print(f'Backed up: {save_dir.name} → {bands_backup.name}')
else:
    print('Backup already exists or save dir not found — skipping.')

nscf_input = PWInput(
    ControlNamelist(calculation='nscf', prefix='mgo_bs',
                    pseudo_dir=str(PSEUDO_DIR), outdir=str(BS_DIR)),
    SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ECUTWFC,
                              nbnd=NBND, occupations='tetrahedra_opt'),
    ElectronsNamelist(),
    AtomicSpeciesCard.from_atoms(atoms, PSEUDOS),
    AtomicPositionsCard.from_atoms(atoms, units='crystal'),
    KPointsAutoCard(2, nk=NK_NSCF),
)

nscf_in  = BS_DIR / 'mgo_bs.nscf.in'
nscf_out = BS_DIR / 'mgo_bs.nscf.out'

if FORCE_RERUN or not nscf_out.exists():
    nscf_in.write_text(nscf_input.to_string())
    with open(nscf_out, 'w') as fh:
        subprocess.run(PW_CMD + ['-input', str(nscf_in)], stdout=fh, stderr=fh, check=True)
print('NSCF done.')

### Total DOS with `dos.x`

The NSCF run has computed eigenvalues $E_{n\mathbf{k}}$ on a uniform $12\times12\times12$ mesh using `occupations = 'tetrahedra_opt'`. Each k-point carries a set of tetrahedron integration weights that replace the Dirac delta in

$$N(E) = \sum_{n,\mathbf{k}} w_{n\mathbf{k}}\,\delta(E - E_{n\mathbf{k}})$$

`dos.x` uses those weights directly — no artificial Gaussian or Lorentzian broadening is needed. The input requires only:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `Emin`, `Emax` | $E_F \pm 20$ eV | energy window |
| `DeltaE` | 0.02 eV | resolution of the output energy grid |

The output file has two columns: energy in eV (absolute, not shifted by $E_F$) and $N(E)$ in states/eV/cell.

In [ ]:
fildos  = str(BS_DIR / 'mgo_bs.dos')
dos_in  = BS_DIR / 'mgo_bs.dos.in'
dos_out = BS_DIR / 'mgo_bs.dos.out'

dos_in.write_text(
    f"&DOS\n  prefix = 'mgo_bs'\n  outdir = '{BS_DIR}'\n"
    f"  fildos = '{fildos}'\n"
    f"  Emin = {E_F - 25:.3f}  Emax = {E_F + 20:.3f}  DeltaE = 0.02\n/\n"
)
if FORCE_RERUN or not Path(fildos).exists():
    with open(dos_in) as fin, open(dos_out, 'w') as fout:
        subprocess.run(DOS_CMD, stdin=fin, stdout=fout, stderr=fout, check=True)

dos_data     = np.loadtxt(fildos, comments='#')
dos_e, dos_n = dos_data[:, 0], dos_data[:, 1]   # eV (absolute), states/eV/cell
print(f'Total DOS: {len(dos_e)} energy points, '
      f'E = [{dos_e[0]:.2f}, {dos_e[-1]:.2f}] eV')

In [ ]:
filpdos = str(BS_DIR / 'mgo_bs')   # projwfc.x appends .pdos_atm#N(X)_wfc#M(l)
pj_in   = BS_DIR / 'mgo_bs.projwfc.in'
pj_out  = BS_DIR / 'mgo_bs.projwfc.out'

pj_namelist = ProjwfcNamelist(
    prefix='mgo_bs', outdir=str(BS_DIR),
    filpdos=filpdos,
    Emin=round(E_F - 25, 3), Emax=round(E_F + 20, 3), DeltaE=0.02,
)
pj_in.write_text(pj_namelist.to_string() + '\n')

if FORCE_RERUN or not list(BS_DIR.glob('mgo_bs.pdos_atm*')):
    with open(pj_in) as fin, open(pj_out, 'w') as fout:
        subprocess.run(PROJWFC_CMD, stdin=fin, stdout=fout, stderr=fout, check=True)

print('PDOS files:')
for f in sorted(BS_DIR.glob('mgo_bs.pdos_atm*')):
    print(f'  {f.name}')

def _load_pdos(filename):
    """Load the PDOS (states/eV/cell) column from a single projwfc.x output file."""
    p = BS_DIR / filename
    return np.loadtxt(p, comments='#')[:, 1] if p.exists() else None

pdos_mg_3s = _load_pdos('mgo_bs.pdos_atm#1(Mg)_wfc#3(s)')
pdos_o_2s  = _load_pdos('mgo_bs.pdos_atm#2(O)_wfc#1(s)')
pdos_o_2p  = _load_pdos('mgo_bs.pdos_atm#2(O)_wfc#2(p)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(dos_e - E_F, dos_n,        color='k',       lw=1.0, label='Total')
if pdos_o_2p  is not None:
    ax.plot(dos_e - E_F, pdos_o_2p,  color='tomato',   lw=1.0, label='O $2p$')
if pdos_o_2s  is not None:
    ax.plot(dos_e - E_F, pdos_o_2s,  color='orange',   lw=1.0, label='O $2s$')
if pdos_mg_3s is not None:
    ax.plot(dos_e - E_F, pdos_mg_3s, color='steelblue',lw=1.0, label='Mg $3s$')

ax.fill_between(dos_e - E_F, dos_n, 0, where=(dos_e <= E_F), alpha=0.12, color='k')
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlim(-25, 15)
ax.set_ylim(bottom=0)
ax.set_xlabel('$E - E_F$ (eV)')
ax.set_ylabel('DOS (states / eV / cell)')
ax.set_title('MgO density of states (DFT-PBE)')
ax.legend()
fig.tight_layout()
plt.show()

## Problem 11: Fat bands `[OPTIONAL]`

**Fat bands** are a band structure plot where each line is drawn with a width proportional to the orbital character of that state. They make the orbital origin of each band immediately visible without switching to a separate PDOS panel.

### Workflow

1. **Restore** the bands wavefunction data (`mgo_bs_bands.save/` → a separate `out/fatbands/` directory, so the NSCF data in `out/bandstructure/` is left untouched).
2. **projwfc.x** on the bands k-path — reads wavefunctions from the bands save, writes `filproj.projwfc_up` with $|\langle\phi_i|\psi_{n\mathbf{k}}\rangle|^2$ for every (band, k-point, orbital) triple.
3. **Parse** `projwfc_up` manually — `qe-tools` cannot parse this file. `read_projwfc_weights` in `bandstructure_tools.py` handles the QE-specific header and per-wavefunction data layout.
4. **Plot** using `LineCollection`: one collection per orbital type, line width ∝ orbital weight.

> [!NOTE]
> `projwfc.x` is run here with only `filproj` set (no `filpdos`, no `kresolveddos`). This writes the raw $|\langle\phi|\psi\rangle|^2$ projections directly, which are exactly the fat-band weights.

In [ ]:
# Use a separate output directory so the NSCF save data is not overwritten
FAT_DIR = RUN_ROOT / 'out' / 'fatbands'
FAT_DIR.mkdir(parents=True, exist_ok=True)

bands_backup = BS_DIR / 'mgo_bs_bands.save'
fat_save     = FAT_DIR / 'mgo_bs.save'
if bands_backup.exists() and not fat_save.exists():
    shutil.copytree(bands_backup, fat_save)
    print(f'Copied bands data → {fat_save}')

filproj    = str(FAT_DIR / 'mgo_bs.proj')
fat_in     = FAT_DIR / 'mgo_bs.fat.in'
fat_out    = FAT_DIR / 'mgo_bs.fat.out'
projwfc_up = Path(filproj + '.projwfc_up')

fat_in.write_text(
    f"&PROJWFC\n  prefix = 'mgo_bs'\n  outdir = '{FAT_DIR}'\n"
    f"  filproj = '{filproj}'\n/\n"
)
if FORCE_RERUN or not projwfc_up.exists():
    with open(fat_in) as fin, open(fat_out, 'w') as fout:
        subprocess.run(PROJWFC_CMD, stdin=fin, stdout=fout, stderr=fout, check=True)

result   = read_projwfc_weights(str(projwfc_up))
weights  = result['weights']    # shape (nkpts, nbnd, natomwfc)
orbitals = result['orbitals']

print(f"Projections: {result['nkpts']} k-pts × {result['nbnd']} bands "
      f"× {result['natomwfc']} atomic wfc")
print('Atomic wavefunctions:')
for orb in orbitals:
    print(f"  wfc#{orb['nwfc']:2d}  atom {orb['na']} ({orb['atom']})  "
          f"n={orb['n']}  l={orb['l']}  m={orb['m']}")

# Build orbital-type index lists
idx_o_p  = [i for i, o in enumerate(orbitals)
            if o['atom'].strip() == 'O'  and o['l'] == 1]
idx_mg_s = [i for i, o in enumerate(orbitals)
            if o['atom'].strip() == 'Mg' and o['l'] == 0]
print(f'\nO 2p  wfc indices: {idx_o_p}')
print(f'Mg 3s wfc indices: {idx_mg_s}')

# Sum over matching orbitals → shape (nbnd, nkpts)  [matches bands_ev]
assert result['nkpts'] == len(k_coords), \
    f"k-point mismatch: projwfc={result['nkpts']}, bands.gnu={len(k_coords)}"
w_o_p  = weights[:, :, idx_o_p ].sum(axis=2).T
w_mg_s = weights[:, :, idx_mg_s].sum(axis=2).T

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

def _fat_lc(k, e_band, w_band, color, scale=15):
    """LineCollection for one band: line width ∝ orbital weight."""
    segs   = [[[k[j], e_band[j]], [k[j+1], e_band[j+1]]]
               for j in range(len(k) - 1)]
    widths = 0.5 * (w_band[:-1] + w_band[1:]) * scale
    return LineCollection(segs, linewidths=widths, colors=color, alpha=0.85)

fig, ax = plt.subplots(figsize=(6, 5))

# Thin grey background bands
for band in bands_ev:
    ax.plot(k_coords, band - E_F, color='lightgray', lw=0.5, zorder=1)

# Fat band overlays
for ibnd in range(len(bands_ev)):
    e = bands_ev[ibnd] - E_F
    if idx_o_p:
        ax.add_collection(_fat_lc(k_coords, e, w_o_p[ibnd],  'tomato',    scale=15))
    if idx_mg_s:
        ax.add_collection(_fat_lc(k_coords, e, w_mg_s[ibnd], 'steelblue', scale=15))

ax.axhline(0, color='gray', lw=0.7, ls='--')
for x in hs_x[1:-1]:
    ax.axvline(x, color='k', lw=0.6)
ax.set_xticks(hs_x)
ax.set_xticklabels(hs_labels)
ax.set_xlim(k_coords[0], k_coords[-1])
ax.set_ylim(-10, 15)
ax.set_ylabel('$E - E_F$ (eV)')
ax.set_title('MgO fat bands (DFT-PBE)')
ax.legend(handles=[
    Line2D([0], [0], color='tomato',    lw=3, label='O $2p$'),
    Line2D([0], [0], color='steelblue', lw=3, label='Mg $3s$'),
])
fig.tight_layout()
plt.show()